# Joint Probability Density Functions: Global Front Analysis


## 1. Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
import xarray as xr
import pandas as pd
from pathlib import Path
import json, re
from scipy.stats import gaussian_kde
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from fronts.properties.colocation import colocate_fronts_with_properties



plt.rcParams['figure.figsize'] = (16, 9)
plt.rcParams['font.size'] = 10
%matplotlib inline

print('Imports OK')

from fronts.properties.jpdf import compute_jpdf, conditional_mean, normalise_by_coriolis


## 2. Load Pre-Computed Results


In [ ]:
run_tag = 'v1_bin_A'
time_str = '2012-11-09_12:00:00'
time_str_safe = time_str.replace(':', '_')

# Load labeled fronts
labeled_fronts = np.load(f'/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived/labeled_fronts_global_{time_str_safe}_{run_tag}.npy')

# Load geometry
df_geom = pd.read_parquet(f'/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived/global_front_geometry_{time_str_safe}_{run_tag}.parquet')

# Load metadata
with open(f'/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived/metadata_{time_str_safe}_{run_tag}.json') as f:
    metadata = json.load(f)

# Load colocation properties
df_coloc = pd.read_parquet(f'/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived/front_properties_{time_str_safe}_{run_tag}.parquet')

# Merge geometry with colocation properties
df_enriched = df_geom.merge(df_coloc, left_on='flabel', right_on='label', how='left')

# Load lat/lon coords
lat = np.load(f'/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived/lat_global_{time_str_safe}_{run_tag}.npy')
lon = np.load(f'/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived/lon_global_{time_str_safe}_{run_tag}.npy')

# Roll longitude if needed (e.g., to -180:180)
lon = np.where(lon > 180, lon - 360, lon)
lon = np.roll(lon, lon.shape[1]//2, axis=1)
labeled_fronts = np.roll(labeled_fronts, labeled_fronts.shape[1]//2, axis=1)

print(f"Loaded {len(df_enriched)} fronts")
print(f"Geometry shape: {labeled_fronts.shape}")
print(f"Lon range: {lon.min():.1f} to {lon.max():.1f}")
print(f"Lat range: {lat.min():.1f} to {lat.max():.1f}")


## 3. Load Physical Property Maps


In [ ]:
properties_dir = '/mnt/tank/Oceanography/data/OGCM/LLC/Fronts/derived'
version        = '1'
timestamp      = '2012-11-09T12_00_00'
property_names = ['gradb2', 'relative_vorticity', 'strain_mag', 'divergence', 'coriolis_f', 'okubo_weiss']
property_files = {
    name: (str(Path(properties_dir) / f'LLC4320_{timestamp}_{name}_v{version}.nc'), name) 
    for name in property_names
}


In [ ]:
import xarray as xr
from fronts.properties.colocation import _load_property_file

property_arrays = {}
for prop_name, src in property_files.items():
    arr = _load_property_file(src)          # handles .nc tuples, .npy, or arrays
    arr = arr.squeeze()                      # drop any singleton time/depth dims
    if downsample_factor:
        arr = arr[::downsample_factor, ::downsample_factor]
    if min_lon_col != 0:                     # same longitude roll as labeled array
        arr = np.roll(arr, shift, axis=1)
    property_arrays[prop_name] = arr
    src_label = src[0] if isinstance(src, (list, tuple)) else (
                    str(src) if not isinstance(src, np.ndarray) else '<array>')
    print(f"  {prop_name:25s}  shape={arr.shape}  "
          f"range=[{float(np.nanmin(arr)):.3g}, {float(np.nanmax(arr)):.3g}]")

print(f'\n✓ {len(property_arrays)} property arrays loaded and aligned')

## 11. Phase-Space: Strain vs |Vorticity|, Colored by Grad-b²

Two-panel hexbin plot on the (strain, |ζ|) phase plane:

* **(a) Global** — all ocean pixels (sub-sampled for speed), each hex colored by the mean grad-b² of the pixels that fall in it.
* **(b) Front co-located** — one point per front (its per-front mean statistics from the co-location step), hexes colored by mean front grad-b².

> Requires `'strain'`, `'relative_vorticity'`, and `'gradb2'` to be present in `property_files` (Cell 5) and therefore in `property_arrays` and `df_enriched`.


In [ ]:
# ─── SETTINGS ────────────────────────────────────────────────────────────────
STRAIN_KEY   = 'strain_mag'             # key in property_arrays / prefix in df_enriched
VORT_KEY     = 'relative_vorticity' # divided by f (signed)
COLOR_KEY    = 'gradb2'             # shown as color (mean per hex)
CORIOLIS_KEY = 'coriolis_f'         # pre-loaded Coriolis field

GRIDSIZE     = 200      # hexbin resolution
N_SAMPLE_GL  = None # max pixels for panel (a); None = use all
COLOR_PCT    = 98      # percentile clip for the color axis
AX_PCT       = 99.8      # percentile clip for x/y axes (based on global+front data)
CMAP         = 'Reds'
LAT_MIN_ABS  = 3.0     # exclude |lat| < this (avoid f ≈ 0 near equator)
# ─────────────────────────────────────────────────────────────────────────────

# ── Availability checks ───────────────────────────────────────────────────────
missing_arr = [k for k in (STRAIN_KEY, VORT_KEY, COLOR_KEY, CORIOLIS_KEY)
               if k not in property_arrays]
if missing_arr:
    raise KeyError(
        f'Keys missing from property_arrays: {missing_arr}\n'
        f'Add them to property_files in Cell 5 and re-run Cell 6.'
    )

col_mean = lambda k: f'{k}_mean'
missing_col = [k for k in (STRAIN_KEY, VORT_KEY, COLOR_KEY, CORIOLIS_KEY)
               if col_mean(k) not in df_enriched.columns]
if missing_col:
    raise KeyError(
        f'Co-located columns missing from df_enriched: '
        f'{[col_mean(k) for k in missing_col]}\n'
        f'Make sure these properties were included before running Cell 8.'
    )

# ── (a) Global arrays — flatten + normalise ───────────────────────────────────
strain_gl_raw = property_arrays[STRAIN_KEY].ravel()
vort_gl_raw   = property_arrays[VORT_KEY].ravel()
color_gl_raw  = property_arrays[COLOR_KEY].ravel()
f_gl          = property_arrays[CORIOLIS_KEY].ravel()

ro_gl    = vort_gl_raw   / f_gl           # ζ / f  (signed Rossby number)
alpha_gl = strain_gl_raw / np.abs(f_gl)   # strain / |f|

lat_gl = lat_global.ravel()
ok_gl  = (np.isfinite(ro_gl) & np.isfinite(alpha_gl) & np.isfinite(color_gl_raw)
          & (np.abs(lat_gl) >= LAT_MIN_ABS))
ro_gl, alpha_gl, color_gl = ro_gl[ok_gl], alpha_gl[ok_gl], color_gl_raw[ok_gl]

if N_SAMPLE_GL and len(ro_gl) > N_SAMPLE_GL:
    idx = np.random.default_rng(42).choice(len(ro_gl), N_SAMPLE_GL, replace=False)
    ro_gl, alpha_gl, color_gl = ro_gl[idx], alpha_gl[idx], color_gl[idx]

print(f'Panel (a): {len(ro_gl):,} global pixels')

# ── (b) Front co-located — per-front mean stats ───────────────────────────────
df_ph = df_enriched[['centroid_lat',
                      col_mean(STRAIN_KEY),
                      col_mean(VORT_KEY),
                      col_mean(COLOR_KEY),
                      col_mean(CORIOLIS_KEY)]].dropna().copy()

f_fr     = df_ph[col_mean(CORIOLIS_KEY)].values
ro_fr    = df_ph[col_mean(VORT_KEY)].values   / f_fr
alpha_fr = df_ph[col_mean(STRAIN_KEY)].values  / np.abs(f_fr)
color_fr = df_ph[col_mean(COLOR_KEY)].values

ok_fr = (np.isfinite(ro_fr) & np.isfinite(alpha_fr) & np.isfinite(color_fr)
         & (np.abs(df_ph['centroid_lat'].values) >= LAT_MIN_ABS))
ro_fr, alpha_fr, color_fr = ro_fr[ok_fr], alpha_fr[ok_fr], color_fr[ok_fr]

print(f'Panel (b): {len(ro_fr):,} co-located fronts')

# ── Shared limits ─────────────────────────────────────────────────────────────
all_color = np.concatenate([color_gl, color_fr])
all_color = all_color[np.isfinite(all_color)]
vmin_c = float(np.percentile(all_color, 100 - COLOR_PCT))
vmax_c = float(np.percentile(all_color, COLOR_PCT))

ro_all  = np.concatenate([ro_gl,    ro_fr])
alp_all = np.concatenate([alpha_gl, alpha_fr])
ro_lim  = float(np.percentile(np.abs(ro_all[np.isfinite(ro_all)]),            AX_PCT))
alp_lim = float(np.percentile(alp_all[np.isfinite(alp_all) & (alp_all > 0)], AX_PCT))
xlim = (-ro_lim, ro_lim)   # ζ/f  — symmetric
ylim = (0.0,     alp_lim)  # strain/|f| — always >= 0

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

datasets = [
    ('(a)  All ocean pixels',            ro_gl,  alpha_gl, color_gl),
    ('(b)  Front co‑located means', ro_fr,  alpha_fr, color_fr),
]

hb_ref = None
for ax, (title, xs, ys, cs) in zip(axes, datasets):
    hb = ax.hexbin(
        xs, ys,
        C=cs,
        gridsize=GRIDSIZE,
        reduce_C_function=np.nanmean,
        cmap=CMAP,
        vmin=vmin_c, vmax=vmax_c,
        linewidths=0.2,
        extent=[xlim[0], xlim[1], ylim[0], ylim[1]],
    )
    hb_ref = hb

    ax.axvline( 0, color='k',    linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(-1, color='gray', linewidth=0.6, linestyle=':',  alpha=0.6)
    ax.axvline( 1, color='gray', linewidth=0.6, linestyle=':',  alpha=0.6)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel(r'$\zeta\,/\,f$  (Rossby number)', fontsize=12)
    ax.set_ylabel(r'Strain $/ |f|$',                   fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.25, linestyle='--')
    ax.text(0.97, 0.03, f'n = {len(xs):,}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

fig.colorbar(hb_ref, ax=axes, shrink=0.75, pad=0.02,
             label=f'Mean  {COLOR_KEY.replace("_", " ").title()}  per hex')
fig.suptitle(
    r'$\zeta/f$  vs  Strain$/|f|$  —  colored by mean '
    + COLOR_KEY.replace('_', ' ').title() + '\n'
    f'Hexbin gridsize = {GRIDSIZE}   |   '
    f'axes clipped to {COLOR_PCT}th percentile   |   '
    f'|lat| ≥ {LAT_MIN_ABS}°',
    fontsize=13, fontweight='bold')
plt.show()
print('\u2713  Phase-space plot done')


## 16. Vorticity–Strain JPDF and Spatial Decomposition

Following [Balwada et al. (2021)](https://doi.org/10.1175/JPO-D-21-0016.1), the flow is decomposed into three dynamical regions using the joint probability distribution function (JPDF) of normalised surface vorticity and strain:

| Region | Definition | Dynamics |
|--------|-----------|----------|
| **AVD** | ζ < 0 and σ < \|ζ\| | Anticyclonic vorticity dominated |
| **CVD** | ζ > 0 and σ < \|ζ\| | Cyclonic vorticity dominated |
| **SD**  | σ ≥ \|ζ\|            | Strain dominated — associated with fronts |

where ζ/f is the Rossby number and σ/|f| is the normalised strain magnitude. Boundaries between regions are the lines σ = |ζ|.

**Plots produced**
- **Fig 1** — JPDF P(ζ/f, σ/|f|) for (a) gridded ocean pixels and (b) front co-located values
- **Fig 2** — Spatial maps analogous to Fig. 4 of Balwada et al.: (b) full ζ/f field, (c) ζ/f in AVD pixels only, (d) ζ/f in CVD pixels only, (e) ζ/f in SD pixels only


In [ ]:
from fronts.properties.jpdf import compute_jpdf, conditional_mean, normalise_by_coriolis
import re as _re

# ─── PROPERTY KEYS ────────────────────────────────────────────────────────────
VORT_KEY_J     = 'relative_vorticity'   # key in property_arrays
STRAIN_KEY_J   = 'strain_mag'
CORIOLIS_KEY_J = 'coriolis_f'

# ─── SUB-REGION SELECTION ─────────────────────────────────────────────────────
# Controls which pixels are used for the JPDF and spatial maps.
#   'region'  → use region_bounds from Section 4
#   'global'  → entire global array (slower; millions of pixels)
#   dict(lat_min=…, lat_max=…, lon_min=…, lon_max=…)  → custom box
JPDF_REGION = 'region'

# ─── JPDF SETTINGS ────────────────────────────────────────────────────────────
N_VORT_BINS   = 100
N_STRAIN_BINS = 80      # only positive half of strain axis
CLIP_PCT      = 99.5    # percentile for auto axis range
VORT_RANGE    = None    # None = auto; or e.g. (-6, 6)
STRAIN_RANGE  = None    # None = auto; or e.g. (0, 6)
MIN_COUNT     = 10      # bins with fewer samples → NaN in conditional means

# Filter df_enriched fronts to the same sub-region as gridded arrays?
FILTER_FRONTS_TO_REGION = True

# ─── LOAD AND NORMALISE GRIDDED ARRAYS ────────────────────────────────────────
for _k in [VORT_KEY_J, STRAIN_KEY_J, CORIOLIS_KEY_J]:
    if _k not in property_arrays:
        raise KeyError(f"{_k!r} not in property_arrays — add it in Section 2.")

vort_gl   = np.asarray(property_arrays[VORT_KEY_J],     dtype=np.float64)
strain_gl = np.asarray(property_arrays[STRAIN_KEY_J],   dtype=np.float64)
coris_gl  = np.asarray(property_arrays[CORIOLIS_KEY_J], dtype=np.float64)

# ─── EQUATORIAL EXCLUSION ─────────────────────────────────────────────────────
# f = 2Ω sin(lat) → 0 at equator, so ζ/f and σ/|f| diverge there.
# Pixels within MIN_ABS_LAT degrees of the equator are set to NaN and
# excluded automatically from the JPDF.  Adjust as needed.
MIN_ABS_LAT = 5.0   # degrees — equatorial band to exclude

ro_gl, snorm_gl = normalise_by_coriolis(
    vort_gl, strain_gl, coris_gl,
    min_abs_lat=MIN_ABS_LAT,
    lat=lat_global,
)
print(f"Normalised — ζ/f  range (0.5–99.5 pct): "
      f"[{np.nanpercentile(ro_gl, 0.5):.2f}, {np.nanpercentile(ro_gl, 99.5):.2f}]")
print(f"           — σ/|f| range (0.5–99.5 pct): "
      f"[{np.nanpercentile(snorm_gl, 0.5):.2f}, {np.nanpercentile(snorm_gl, 99.5):.2f}]")

# ─── SUB-REGION MASK ──────────────────────────────────────────────────────────
if JPDF_REGION == 'region':
    _bounds = region_bounds
elif isinstance(JPDF_REGION, dict):
    _bounds = JPDF_REGION
else:
    _bounds = None   # global

if _bounds is not None:
    sub_mask_j = (
        (lat_global >= _bounds['lat_min']) & (lat_global <= _bounds['lat_max']) &
        (lon_global >= _bounds['lon_min']) & (lon_global <= _bounds['lon_max'])
    )
    ro_sub    = np.where(sub_mask_j, ro_gl,    np.nan)
    snorm_sub = np.where(sub_mask_j, snorm_gl, np.nan)
    print(f"\nSub-region {_bounds}: {sub_mask_j.sum():,} pixels")
else:
    sub_mask_j = np.isfinite(ro_gl) & np.isfinite(snorm_gl)
    ro_sub     = ro_gl.copy()
    snorm_sub  = snorm_gl.copy()
    print("\nUsing full global arrays.")

# Tight row/col crop for spatial maps (avoids plotting huge NaN-filled regions)
_valid_rc = np.where(sub_mask_j)
_r0, _r1  = _valid_rc[0].min(), _valid_rc[0].max() + 1
_c0, _c1  = _valid_rc[1].min(), _valid_rc[1].max() + 1
lat_crop  = lat_global[_r0:_r1, _c0:_c1]
lon_crop  = lon_global[_r0:_r1, _c0:_c1]
ro_crop   = ro_sub[    _r0:_r1, _c0:_c1]
snorm_crop = snorm_sub[_r0:_r1, _c0:_c1]
print(f"Spatial crop: rows {_r0}:{_r1}, cols {_c0}:{_c1}  → shape {ro_crop.shape}")

# ─── JPDF: GRIDDED CASE ───────────────────────────────────────────────────────
print("\nComputing JPDF (gridded)…")
jpdf_grid = compute_jpdf(
    ro_sub, snorm_sub,
    n_vort_bins=N_VORT_BINS, n_strain_bins=N_STRAIN_BINS,
    vort_range=VORT_RANGE, strain_range=STRAIN_RANGE, clip_pct=CLIP_PCT,
)
_fg = jpdf_grid.region_fractions()
print(f"  n_samples={jpdf_grid.n_samples:,}  "
      f"AVD={_fg['AVD']:.3f}  CVD={_fg['CVD']:.3f}  SD={_fg['SD']:.3f}")

# ─── JPDF: FRONT CO-LOCATED CASE ─────────────────────────────────────────────
def _col(df, base, sfx=('_median', '_mean', '')):
    for s in sfx:
        if base + s in df.columns:
            return base + s
    return None

_vcol = _col(df_enriched, VORT_KEY_J)
_scol = _col(df_enriched, STRAIN_KEY_J)
_ccol = _col(df_enriched, CORIOLIS_KEY_J)

jpdf_fronts = None
if None in (_vcol, _scol, _ccol):
    print(f"\n⚠ Front JPDF skipped — columns not found: "
          f"vort={_vcol}, strain={_scol}, coriolis={_ccol}")
else:
    df_j = df_enriched.copy()
    if FILTER_FRONTS_TO_REGION and _bounds is not None:
        df_j = df_j[
            (df_j['centroid_lat'] >= _bounds['lat_min']) &
            (df_j['centroid_lat'] <= _bounds['lat_max']) &
            (df_j['centroid_lon'] >= _bounds['lon_min']) &
            (df_j['centroid_lon'] <= _bounds['lon_max'])
        ]
    # Exclude equatorial fronts (|centroid_lat| < MIN_ABS_LAT) where f ≈ 0
    df_j = df_j[df_j['centroid_lat'].abs() >= MIN_ABS_LAT]
    ro_fr    = df_j[_vcol] / df_j[_ccol]
    snorm_fr = df_j[_scol] / df_j[_ccol].abs()
    # Guard against any residual near-zero coriolis values
    _f_min_fr = 2 * 7.2921e-5 * np.sin(np.deg2rad(MIN_ABS_LAT))
    _valid_fr = df_j[_ccol].abs() >= _f_min_fr
    ro_fr    = ro_fr[_valid_fr]
    snorm_fr = snorm_fr[_valid_fr]
    df_j     = df_j[_valid_fr]
    # Share bin edges with gridded JPDF for direct comparison
    print(f"\nComputing JPDF (fronts, n={len(df_j):,})…")
    jpdf_fronts = compute_jpdf(
        ro_fr, snorm_fr,
        vort_range  = (float(jpdf_grid.vort_edges[0]),   float(jpdf_grid.vort_edges[-1])),
        strain_range= (float(jpdf_grid.strain_edges[0]), float(jpdf_grid.strain_edges[-1])),
        n_vort_bins=N_VORT_BINS, n_strain_bins=N_STRAIN_BINS,
    )
    _ff = jpdf_fronts.region_fractions()
    print(f"  n_samples={jpdf_fronts.n_samples:,}  "
          f"AVD={_ff['AVD']:.3f}  CVD={_ff['CVD']:.3f}  SD={_ff['SD']:.3f}")

# ─── SPATIAL DECOMPOSITION (on cropped sub-region) ───────────────────────────
# Assign each pixel to AVD / CVD / SD based on normalised vorticity and strain
_fin = np.isfinite(ro_crop) & np.isfinite(snorm_crop)
avd_pix = (ro_crop < 0) & (snorm_crop < np.abs(ro_crop)) & _fin
cvd_pix = (ro_crop > 0) & (snorm_crop < np.abs(ro_crop)) & _fin
sd_pix  = (snorm_crop >= np.abs(ro_crop))                 & _fin

vort_map_full = ro_crop.copy()                            # (b) full ζ/f
vort_map_avd  = np.where(avd_pix, ro_crop, np.nan)       # (c) AVD pixels only
vort_map_cvd  = np.where(cvd_pix, ro_crop, np.nan)       # (d) CVD pixels only
vort_map_sd   = np.where(sd_pix,  ro_crop, np.nan)       # (e) SD pixels only

_n = _fin.sum()
print(f"\nSpatial decomposition ({_n:,} valid pixels):")
print(f"  AVD: {avd_pix.sum():,} ({avd_pix.sum()/_n*100:.1f}%)")
print(f"  CVD: {cvd_pix.sum():,} ({cvd_pix.sum()/_n*100:.1f}%)")
print(f"  SD : {sd_pix.sum():,}  ({sd_pix.sum()/_n*100:.1f}%)")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm, TwoSlopeNorm
import numpy as np

# ─── SETTINGS ─────────────────────────────────────────────────────────────────
JPDF_CMAP    = 'Reds'
VORT_CMAP    = 'RdBu_r'
MAP_PCT      = 99.5     # percentile for shared vorticity colorscale

# ─── HELPER: JPDF axes ────────────────────────────────────────────────────────
def _jpdf_panel(ax, jpdf_res, title):
    """Plot P(ζ/f, σ/|f|) with σ=|ζ| region boundaries."""
    ve = jpdf_res.vort_edges
    se = jpdf_res.strain_edges
    pdf_plot = jpdf_res.pdf.T   # transpose: rows=strain (y), cols=vorticity (x)
    pdf_min  = pdf_plot[pdf_plot > 0].min()

    pm = ax.pcolormesh(ve, se, pdf_plot,
                       norm=LogNorm(vmin=pdf_min, vmax=pdf_plot.max()),
                       cmap=JPDF_CMAP, rasterized=True)
    plt.colorbar(pm, ax=ax, label=r'$P(\zeta/f,\;\sigma/|f|)$', fraction=0.046, pad=0.04)

    # σ = |ζ| boundary lines
    x_line = np.linspace(ve[0], ve[-1], 300)
    ax.plot(x_line,  np.abs(x_line), color='gray', lw=1.0, ls='--', alpha=0.9, zorder=5)
    ax.axvline(0,    color='gray',   lw=0.6, ls='-',  alpha=0.5, zorder=5)

    # Region labels
    _vspan = abs(ve[-1] - ve[0])
    _speak = se[-1] * 0.75
    _vlow  = se[-1] * 0.25
    ax.text(-_vspan * 0.3, _vlow,  'AVD', color='dimgray', fontsize=9, ha='center')
    ax.text( _vspan * 0.3, _vlow,  'CVD', color='dimgray', fontsize=9, ha='center')
    ax.text(0,             _speak, 'SD',  color='dimgray', fontsize=9, ha='center')

    fracs = jpdf_res.region_fractions()
    ax.set_title(
        f'{title}  (n={jpdf_res.n_samples:,})\n'
        f'AVD {fracs["AVD"]:.2f}  |  CVD {fracs["CVD"]:.2f}  |  SD {fracs["SD"]:.2f}',
        fontsize=10)
    ax.set_xlabel(r'$\zeta\,/\,f$  (Rossby number)', fontsize=10)
    ax.set_ylabel(r'$\sigma\,/\,|f|$',               fontsize=10)
    ax.set_xlim(ve[0],  ve[-1])
    ax.set_ylim(se[0],  se[-1])

# ── Figure 1: JPDF plots ──────────────────────────────────────────────────────
_ncols = 2 if jpdf_fronts is not None else 1
fig1, _ax1 = plt.subplots(1, _ncols, figsize=(6.5 * _ncols, 5.5))
if _ncols == 1:
    _ax1 = [_ax1]

_region_str = (f"{_bounds['lat_min']:.0f}°–{_bounds['lat_max']:.0f}°N, "
               f"{_bounds['lon_min']:.0f}°–{_bounds['lon_max']:.0f}°E"
               if _bounds else 'Global')

_jpdf_panel(_ax1[0], jpdf_grid,   f'(a) Gridded ocean')
if jpdf_fronts is not None:
    _jpdf_panel(_ax1[1], jpdf_fronts, f'(b) Front co-located')

fig1.suptitle(f'Vorticity–Strain JPDF  [{_region_str}]', fontsize=12, y=1.01)
fig1.tight_layout()
plt.show()

# ── Figure 2: Spatial decomposition maps (Fig. 4, Balwada et al.) ─────────────
_map_panels = [
    (vort_map_full, r'(b) $\zeta/f$  — full field'),
    (vort_map_avd,  r'(c) $\zeta/f$  — AVD region  (ζ<0, σ<|ζ|)'),
    (vort_map_cvd,  r'(d) $\zeta/f$  — CVD region  (ζ>0, σ<|ζ|)'),
    (vort_map_sd,   r'(e) $\zeta/f$  — SD region   (σ≥|ζ|, fronts)'),
]

# Shared colorscale: ±98th-percentile of full field
_vabs = float(np.nanpercentile(np.abs(vort_map_full), MAP_PCT))
_norm = TwoSlopeNorm(vcenter=0, vmin=-_vabs, vmax=_vabs)

fig2, _ax2 = plt.subplots(2, 2, figsize=(16, 11))
_ax2 = _ax2.ravel()

for _ax, (_data, _title) in zip(_ax2, _map_panels):
    _pm = _ax.pcolormesh(lon_crop, lat_crop, np.ma.masked_invalid(_data),
                         norm=_norm, cmap=VORT_CMAP,
                         shading='auto', rasterized=True)
    plt.colorbar(_pm, ax=_ax, fraction=0.035, pad=0.02,
                 label=r'$\zeta\,/\,f$')
    _ax.set_xlim(lon_crop.min(), lon_crop.max())
    _ax.set_ylim(lat_crop.min(), lat_crop.max())
    _ax.set_xlabel('Longitude', fontsize=9)
    _ax.set_ylabel('Latitude',  fontsize=9)
    _ax.set_title(_title, fontsize=11)

# ─── FRONT OVERLAY on panel (e) — SD region ──────────────────────────────────
labeled_crop = labeled_global[_r0:_r1, _c0:_c1]
_front_ov    = np.where(labeled_crop > 0, 1.0, np.nan)
from matplotlib.colors import ListedColormap as _LC
_ax2[3].pcolormesh(lon_crop, lat_crop, np.ma.masked_invalid(_front_ov),
                   cmap=_LC(['black']), vmin=0, vmax=1,
                   alpha=1, shading='auto', rasterized=True, zorder=4)
_ax2[3].set_title(_ax2[3].get_title() + '  + fronts', fontsize=11)


fig2.suptitle(
    r'Spatial decomposition of $\zeta/f$ into vorticity–strain JPDF regions'
    f'\n[{_region_str}]', fontsize=12)
fig2.tight_layout()
plt.show()


## 4. Divergence in Phase Space


In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm, TwoSlopeNorm
import numpy as np

# ─── SETTINGS ─────────────────────────────────────────────────────────────────
DIV_KEY_J  = 'divergence'   # key in property_arrays

# Target δ/|f| values for each sub-panel.
#   None = auto (25th / 50th / 75th percentile of each sign's population).
#   Override with explicit lists, e.g.:
#     CONV_LEVELS = [-0.3, -0.6, -1.2]   # must be negative, weakest → strongest
#     DIV_LEVELS  = [ 0.3,  0.6,  1.2]   # must be positive, weakest → strongest
CONV_LEVELS = [-0.1, -0.6, -1.0]
DIV_LEVELS  = [0.1, 0.6, 1.0]

# Half-bandwidth ± around each target value used to select pixels.
#   None = auto (one-quarter of the spacing between adjacent levels).
LEVEL_BW    = None

DIV_CMAP      = 'RdBu_r'   # diverging: blue = convergence, red = divergence
JPDF_SUB_CMAP = 'Reds'     # shared colormap for sub-JPDF panels

# ─── LOAD AND NORMALISE DIVERGENCE ────────────────────────────────────────────
if DIV_KEY_J not in property_arrays:
    raise KeyError(f"{DIV_KEY_J!r} not in property_arrays — add it in Section 2.")

div_gl  = np.asarray(property_arrays[DIV_KEY_J], dtype=np.float64)
div_sub = np.where(sub_mask_j, div_gl, np.nan)

_f_min_div  = 2 * 7.2921e-5 * np.sin(np.deg2rad(MIN_ABS_LAT))
coris_sub_d = np.where(sub_mask_j, coris_gl, np.nan)
coris_abs_d = np.where(np.abs(coris_sub_d) >= _f_min_div, np.abs(coris_sub_d), np.nan)
div_norm    = div_sub / coris_abs_d
div_norm    = np.where(np.isfinite(ro_sub), div_norm, np.nan)

print(f"δ/|f|  0.5/99.5 pct: "
      f"[{np.nanpercentile(div_norm, 0.5):.3f}, {np.nanpercentile(div_norm, 99.5):.3f}]")

# ─── (a) CONDITIONAL MEAN OF δ/|f| ───────────────────────────────────────────
print("Computing conditional mean of δ/|f|…")
cond_div = conditional_mean(
    div_norm, ro_sub, snorm_sub,
    field_name   = 'div_norm',
    vort_edges   = jpdf_grid.vort_edges,
    strain_edges = jpdf_grid.strain_edges,
    min_count    = MIN_COUNT,
)

# ─── AUTO-COMPUTE LEVELS AND BANDWIDTH ───────────────────────────────────────
_valid    = div_norm[np.isfinite(div_norm) & np.isfinite(ro_sub) & np.isfinite(snorm_sub)]
_conv_val = _valid[_valid < 0]
_divg_val = _valid[_valid > 0]

if CONV_LEVELS is None:
    CONV_LEVELS = [float(np.percentile(_conv_val, p)) for p in [75, 50, 25]]
    # 75th pct of negatives = weakest conv; 25th pct = strongest conv
if DIV_LEVELS is None:
    DIV_LEVELS  = [float(np.percentile(_divg_val, p)) for p in [25, 50, 75]]
    # 25th pct of positives = weakest div; 75th pct = strongest div

all_levels = CONV_LEVELS + DIV_LEVELS   # 6 values: conv weak→strong, div weak→strong

if LEVEL_BW is None:
    _spacings  = [abs(all_levels[i+1] - all_levels[i]) for i in range(len(all_levels)-1)]
    LEVEL_BW   = min(_spacings) * 0.35   # ±35 % of smallest inter-level gap

print(f"Convergent levels : {[f'{v:.3f}' for v in CONV_LEVELS]}")
print(f"Divergent  levels : {[f'{v:.3f}' for v in DIV_LEVELS]}")
print(f"Bandwidth ±       : {LEVEL_BW:.4f}")

# ─── COMPUTE SUB-JPDFs FOR EACH TARGET LEVEL ─────────────────────────────────
_vr = (float(jpdf_grid.vort_edges[0]),   float(jpdf_grid.vort_edges[-1]))
_sr = (float(jpdf_grid.strain_edges[0]), float(jpdf_grid.strain_edges[-1]))
_nv = jpdf_grid.pdf.shape[0]
_ns = jpdf_grid.pdf.shape[1]

panel_ids     = list('bcdefg')
jpdf_sub_list = []

print("\nComputing sub-JPDFs…")
for level, pid in zip(all_levels, panel_ids):
    _mask = (div_norm >= level - LEVEL_BW) & (div_norm <= level + LEVEL_BW)
    _j    = compute_jpdf(
        np.where(_mask, ro_sub,    np.nan),
        np.where(_mask, snorm_sub, np.nan),
        n_vort_bins=_nv, n_strain_bins=_ns,
        vort_range=_vr,  strain_range=_sr,
    )
    jpdf_sub_list.append(_j)
    _sign = 'conv' if level < 0 else 'div'
    print(f"  ({pid})  δ/|f| ≈ {level:+.3f} ± {LEVEL_BW:.3f}  "
          f"[{_sign}]   n = {_j.n_samples:,}")

_sub_pdfs = [j.pdf for j in jpdf_sub_list]
_pdf_max  = max(p.max() for p in _sub_pdfs)
_pdf_min  = min(p[p > 0].min() for p in _sub_pdfs if (p > 0).any())


In [ ]:

# ─── PLOT ─────────────────────────────────────────────────────────────────────
# Layout: (a) spans full top row; (b-d) convergent in row 1; (e-g) divergent in row 2
fig = plt.figure(figsize=(26, 13))
gs  = GridSpec(4, 3, figure=fig, wspace=0.38, hspace=0.52,
               height_ratios=[1, 1, 1, 1])

ax_a   = fig.add_subplot(gs[:2, :2])                     # rows 0+1, full width
ax_top = [fig.add_subplot(gs[2, c]) for c in range(3)]  # (b)(c)(d)
ax_bot = [fig.add_subplot(gs[3, c]) for c in range(3)]  # (e)(f)(g)
sub_axes = ax_top + ax_bot

# ── (a) Conditional mean ──────────────────────────────────────────────────────
_cd   = cond_div.mean
_cabs = float(np.nanpercentile(np.abs(_cd[np.isfinite(_cd)]), 99.9))
ve, se = cond_div.vort_edges, cond_div.strain_edges

_pm_a = ax_a.pcolormesh(ve, se, np.ma.masked_invalid(_cd.T),
                         norm=TwoSlopeNorm(vcenter=0, vmin=-_cabs, vmax=_cabs),
                         cmap=DIV_CMAP, rasterized=True)
plt.colorbar(_pm_a, ax=ax_a,
             label=r'$\overline{\delta/|f|}^{\,\zeta\sigma}$',
             fraction=0.025, pad=0.02)

# Mark the 6 target levels as vertical dashed lines on the colorbar — not feasible,
# but add tick-like annotations along the bottom of panel (a)
_xl = np.linspace(ve[0], ve[-1], 300)
ax_a.plot(_xl, np.abs(_xl), color='k', lw=0.9, ls='--', alpha=0.55, zorder=5)
ax_a.axvline(0, color='k', lw=0.6, ls='-', alpha=0.35, zorder=5)

_vs = abs(ve[-1] - ve[0])
for _t, _x, _y in [('AVD', -_vs*0.28, se[-1]*0.25),
                    ('CVD',  _vs*0.28, se[-1]*0.25),
                    ('SD',   0,        se[-1]*0.78)]:
    ax_a.text(_x, _y, _t, color='k', fontsize=9, ha='center', alpha=0.55)

ax_a.set_xlabel(r'$\zeta\,/\,f$', fontsize=10)
ax_a.set_ylabel(r'$\sigma\,/\,|f|$', fontsize=10)
ax_a.set_xlim(ve[0], ve[-1])
ax_a.set_ylim(se[0], se[-1])
ax_a.set_title(r'(a)  Conditional mean  $\overline{\delta/|f|}^{\,\zeta\sigma}$',
               fontsize=11)

# ── (b–g) Sub-JPDFs ──────────────────────────────────────────────────────────
_norm_sub = LogNorm(vmin=_pdf_min, vmax=_pdf_max)

for ax, _j, level, pid in zip(sub_axes, jpdf_sub_list, all_levels, panel_ids):
    ax.pcolormesh(_j.vort_edges, _j.strain_edges, _j.pdf.T,
                  norm=_norm_sub, cmap=JPDF_SUB_CMAP, rasterized=True)

    _xl2 = np.linspace(_j.vort_edges[0], _j.vort_edges[-1], 300)
    ax.plot(_xl2, np.abs(_xl2), color='gray', lw=0.8, ls='--', alpha=0.8, zorder=5)
    ax.axvline(0, color='gray', lw=0.5, ls='-', alpha=0.5, zorder=5)

    _sign = 'conv.' if level < 0 else 'div.'
    ax.set_title(
        f'({pid})  {_sign}   '
        r'$\delta/|f|$'
        f' $\\approx$ {level:+.2f}  ($\\pm${LEVEL_BW:.2f})\n'
        f'$n = {{{_j.n_samples:,}}}$',
        fontsize=9)
    ax.set_xlabel(r'$\zeta/f$', fontsize=9)
    ax.set_ylabel(r'$\sigma/|f|$', fontsize=9)
    ax.set_xlim(_j.vort_edges[0],   _j.vort_edges[-1])
    ax.set_ylim(_j.strain_edges[0], _j.strain_edges[-1])
    ax.tick_params(labelsize=8)

# Shared colorbar for sub-panels
_sm = plt.cm.ScalarMappable(cmap=JPDF_SUB_CMAP, norm=_norm_sub)
_sm.set_array([])
fig.colorbar(_sm, ax=sub_axes, label=r'$P(\zeta/f,\;\sigma/|f|)$',
             fraction=0.018, pad=0.02, shrink=0.9)

fig.suptitle(
    r'Divergence in $(\zeta/f,\,\sigma/|f|)$ phase space'
    '\n'
    r'(a) conditional mean; (b–g) JPDF at target $\delta/|f|$ values'
    f'   [{_region_str}]',
    fontsize=12, y=1.01)
plt.show()


## Section 18 — Generalised Conditional Mean in Vorticity–Strain Space

Set `FIELD_KEY` to any key present in `property_arrays` (and optionally in `df_enriched`) to compute and visualise its conditional mean $\overline{F}^{\zeta\sigma}(\zeta/f,\,\sigma/|f|)$.  
The colorscale switches automatically between **LogNorm + sequential** (positive-only fields) and **TwoSlopeNorm + diverging** (signed fields such as Okubo–Weiss).


In [ ]:
# ─── SETTINGS ─────────────────────────────────────────────────────────────────
# Change these two lines to switch to any other field in property_arrays.
FIELD_KEY   = 'okubo_weiss'      # key in property_arrays / column prefix in df_enriched
FIELD_LABEL = r'$OW$'            # LaTeX label used in colorbar and titles
# ──────────────────────────────────────────────────────────────────────────────

# ─── GRIDDED CASE ─────────────────────────────────────────────────────────────
if FIELD_KEY not in property_arrays:
    raise KeyError(f"{FIELD_KEY!r} not in property_arrays — add it in Section 2.")

field_gl  = np.asarray(property_arrays[FIELD_KEY], dtype=np.float64)
field_sub = np.where(sub_mask_j, field_gl, np.nan)   # same sub-region as Sec 16

cond_field_grid = conditional_mean(
    field_sub, ro_sub, snorm_sub,
    field_name   = FIELD_KEY,
    vort_edges   = jpdf_grid.vort_edges,
    strain_edges = jpdf_grid.strain_edges,
    min_count    = MIN_COUNT,
)
_fg = np.isfinite(cond_field_grid.mean)
print(f"Gridded — {_fg.sum()} bins  ({_fg.sum() / cond_field_grid.mean.size * 100:.1f}%)")
print(f"  Range: [{np.nanmin(cond_field_grid.mean):.3e}, {np.nanmax(cond_field_grid.mean):.3e}]")

# ─── FRONT CO-LOCATED CASE ────────────────────────────────────────────────────
cond_field_fronts = None
if jpdf_fronts is not None:
    _fcol = _col(df_enriched, FIELD_KEY)
    if _fcol is None:
        print(f"\n\u26a0 Front conditional mean skipped — no column matching {FIELD_KEY!r}")
    else:
        field_fr = df_j.loc[_valid_fr, _fcol] if _valid_fr.any() else df_j[_fcol]
        cond_field_fronts = conditional_mean(
            field_fr, ro_fr, snorm_fr,
            field_name   = FIELD_KEY,
            vort_edges   = jpdf_fronts.vort_edges,
            strain_edges = jpdf_fronts.strain_edges,
            min_count    = MIN_COUNT,
        )
        _ff = np.isfinite(cond_field_fronts.mean)
        print(f"\nFronts  — {_ff.sum()} bins  ({_ff.sum() / cond_field_fronts.mean.size * 100:.1f}%)")
        print(f"  Range: [{np.nanmin(cond_field_fronts.mean):.3e}, {np.nanmax(cond_field_fronts.mean):.3e}]")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm
import numpy as np

# ─── SETTINGS ─────────────────────────────────────────────────────────────────
COND_PCT_GEN     = 99.5   # symmetric percentile for colorscale clipping
JPDF_OVERLAY_GEN = True   # overlay JPDF contours on the conditional mean plot
# ──────────────────────────────────────────────────────────────────────────────

# ─── AUTO-DETECT COLORSCALE ───────────────────────────────────────────────────
_all_vals_gen = []
for _c in [cond_field_grid, cond_field_fronts]:
    if _c is not None:
        _v = _c.mean[np.isfinite(_c.mean)]
        if len(_v):
            _all_vals_gen.extend(_v.tolist())
_all_vals_gen = np.array(_all_vals_gen)

_has_neg = bool(np.any(_all_vals_gen < 0))

if _has_neg:
    # Signed field (e.g. Okubo-Weiss): diverging colormap + TwoSlopeNorm
    _CMAP_GEN  = 'RdBu_r'
    _abs_max   = float(np.percentile(np.abs(_all_vals_gen), COND_PCT_GEN))
    _norm_gen  = TwoSlopeNorm(vmin=-_abs_max, vcenter=0.0, vmax=_abs_max)
    _line_col  = 'k'
    _label_col = 'k'
    _vmin_gen, _vmax_gen = -_abs_max, _abs_max
else:
    # Positive-only field (e.g. grad-b²): sequential + LogNorm
    _CMAP_GEN  = 'magma'
    _pos_vals  = _all_vals_gen[_all_vals_gen > 0]
    _vmin_gen  = float(np.percentile(_pos_vals, 100 - COND_PCT_GEN))
    _vmax_gen  = float(np.percentile(_pos_vals, COND_PCT_GEN))
    _norm_gen  = LogNorm(vmin=_vmin_gen, vmax=_vmax_gen)
    _line_col  = 'white'
    _label_col = 'white'

print(f"Field: {FIELD_KEY}  |  signed={_has_neg}  |  cmap={_CMAP_GEN}")
# strip outer dollar signs so FIELD_LABEL can be embedded in math mode
_fl_math = FIELD_LABEL.strip("$")
print(f"Color range: [{_vmin_gen:.3e}, {_vmax_gen:.3e}]")

# ─── HELPER: conditional mean panel ───────────────────────────────────────────
def _cond_panel_gen(ax, cond_res, jpdf_res, title):
    """Plot generalised conditional mean with auto norm/cmap."""
    ve   = cond_res.vort_edges
    se   = cond_res.strain_edges
    data = cond_res.mean.T

    pm = ax.pcolormesh(ve, se, np.ma.masked_invalid(data),
                       norm=_norm_gen, cmap=_CMAP_GEN, rasterized=True)
    plt.colorbar(pm, ax=ax,
                 label=rf'$\overline{{{_fl_math}}}^{{\,\zeta\sigma}}$',
                 fraction=0.046, pad=0.04)

    # Optional JPDF contours
    if JPDF_OVERLAY_GEN and jpdf_res is not None:
        _pdf_p = jpdf_res.pdf.T
        _lev = np.logspace(np.log10(_pdf_p[_pdf_p > 0].min()),
                           np.log10(_pdf_p.max()), 6)[1:-1]
        ax.contour(jpdf_res.vort_centers, jpdf_res.strain_centers,
                   _pdf_p, levels=_lev, colors=_line_col,
                   linewidths=0.6, alpha=0.5, zorder=5)

    # sigma = |zeta| boundary
    x_line = np.linspace(ve[0], ve[-1], 300)
    ax.plot(x_line, np.abs(x_line), color=_line_col, lw=1.0, ls='--',
            alpha=0.8, zorder=6)
    ax.axvline(0, color=_line_col, lw=0.6, ls='-', alpha=0.4, zorder=6)

    # Region labels
    _vspan = abs(ve[-1] - ve[0])
    _speak = se[-1] * 0.75
    _vlow  = se[-1] * 0.25
    for _txt, _x, _y in [('AVD', -_vspan * 0.3, _vlow),
                          ('CVD',  _vspan * 0.3, _vlow),
                          ('SD',   0,            _speak)]:
        ax.text(_x, _y, _txt, color=_label_col, fontsize=9, ha='center',
                fontweight='bold', alpha=0.85)

    fracs = jpdf_res.region_fractions() if jpdf_res is not None else {}
    _frac_str = (f'AVD {fracs.get("AVD", 0):.2f}  |  CVD {fracs.get("CVD", 0):.2f}'
                 f'  |  SD {fracs.get("SD", 0):.2f}') if fracs else ''
    ax.set_title(f'{title}\n{_frac_str}', fontsize=10)
    ax.set_xlabel(r'$\zeta\,/\,f$  (Rossby number)', fontsize=10)
    ax.set_ylabel(r'$\sigma\,/\,|f|$',               fontsize=10)
    ax.set_xlim(ve[0], ve[-1])
    ax.set_ylim(se[0], se[-1])


# ─── FIGURE 1: Conditional mean panel(s) ──────────────────────────────────────
_ncols_gen = 2 if cond_field_fronts is not None else 1
fig1, _ax1 = plt.subplots(1, _ncols_gen, figsize=(6.5 * _ncols_gen, 5.5))
if _ncols_gen == 1:
    _ax1 = [_ax1]

_region_str_gen = (
    f"{_bounds['lat_min']:.0f}\u00b0\u2013{_bounds['lat_max']:.0f}\u00b0N, "
    f"{_bounds['lon_min']:.0f}\u00b0\u2013{_bounds['lon_max']:.0f}\u00b0E"
    if _bounds else 'Global')

_cond_panel_gen(_ax1[0], cond_field_grid,   jpdf_grid,   '(a) Gridded ocean')
if cond_field_fronts is not None:
    _cond_panel_gen(_ax1[1], cond_field_fronts, jpdf_fronts, '(b) Front co-located')

fig1.suptitle(
    rf'Conditional mean of {FIELD_LABEL} in vorticity\u2013strain space'
    f'\n[{_region_str_gen}]  \u2014  contours = JPDF',
    fontsize=12, y=1.01)
fig1.tight_layout()
plt.show()

# ─── FIGURE 2: JPDF and conditional mean side by side ─────────────────────────
fig2, _ax2 = plt.subplots(1, 2, figsize=(13, 5.5))

_j = jpdf_grid
_pdf_plot_gen = _j.pdf.T

_pm1 = _ax2[0].pcolormesh(_j.vort_edges, _j.strain_edges, _pdf_plot_gen,
                           norm=LogNorm(vmin=_pdf_plot_gen[_pdf_plot_gen > 0].min(),
                                        vmax=_pdf_plot_gen.max()),
                           cmap='Reds', rasterized=True)
plt.colorbar(_pm1, ax=_ax2[0],
             label=r'$P(\zeta/f,\;\sigma/|f|)$',
             fraction=0.046, pad=0.04)

if JPDF_OVERLAY_GEN:
    _lev_j = np.logspace(np.log10(_pdf_plot_gen[_pdf_plot_gen > 0].min()),
                         np.log10(_pdf_plot_gen.max()), 8)[1:-1]
    _ax2[0].contour(_j.vort_centers, _j.strain_centers, _pdf_plot_gen,
                    levels=_lev_j, colors='black',
                    linewidths=0.5, alpha=0.45, zorder=5)

_c = cond_field_grid
_pm2 = _ax2[1].pcolormesh(_c.vort_edges, _c.strain_edges,
                           np.ma.masked_invalid(_c.mean.T),
                           norm=_norm_gen, cmap=_CMAP_GEN, rasterized=True)
plt.colorbar(_pm2, ax=_ax2[1],
             label=rf'$\overline{{{_fl_math}}}^{{\,\zeta\sigma}}$',
             fraction=0.046, pad=0.04)

if JPDF_OVERLAY_GEN:
    _lev_c = np.logspace(np.log10(_pdf_plot_gen[_pdf_plot_gen > 0].min()),
                         np.log10(_pdf_plot_gen.max()), 8)[1:-1]
    _ax2[1].contour(_j.vort_centers, _j.strain_centers, _pdf_plot_gen,
                    levels=_lev_c, colors=_line_col,
                    linewidths=0.6, alpha=0.5, zorder=5)

_gray = 'gray'
for _ax, _ttl in zip(_ax2,
                     [r'(a) JPDF  $P(\zeta/f,\;\sigma/|f|)$',
                      rf'(b) Cond. mean  {FIELD_LABEL}']):
    x_line = np.linspace(_j.vort_edges[0], _j.vort_edges[-1], 300)
    _ax.plot(x_line, np.abs(x_line), color=_gray, lw=1.0, ls='--', alpha=0.85)
    _ax.axvline(0, color=_gray, lw=0.6, ls='-', alpha=0.5)
    _ax.set_xlabel(r'$\zeta\,/\,f$', fontsize=10)
    _ax.set_ylabel(r'$\sigma\,/\,|f|$', fontsize=10)
    _ax.set_xlim(_j.vort_edges[0], _j.vort_edges[-1])
    _ax.set_ylim(_j.strain_edges[0], _j.strain_edges[-1])
    _ax.set_title(_ttl, fontsize=11)
    for _txt, _x, _y in [('AVD', -abs(_j.vort_edges[-1]) * 0.5, _j.strain_edges[-1] * 0.25),
                          ('CVD',  abs(_j.vort_edges[-1]) * 0.5, _j.strain_edges[-1] * 0.25),
                          ('SD',   0,                             _j.strain_edges[-1] * 0.75)]:
        _ax.text(_x, _y, _txt, color=_gray, fontsize=8, ha='center', alpha=0.9)

fig2.suptitle(
    rf'JPDF vs conditional mean of {FIELD_LABEL}  [{_region_str_gen}]',
    fontsize=12)
fig2.tight_layout()
plt.show()
